# NB6 — Agentic Retrieval: truy xuất như một *tool*

**Stack:** `app.agent` (tool schema + planner + reflection) trên `FilteredIndex`.
Maps to deck §6 "Retrieval Như Một Tool" + "Ghép Ngữ Cảnh".

> Trong RAG cổ điển, retrieval là một *bước* trong pipeline. Với agent, nó là
> một **tool** mà model tự quyết định gọi — gọi mấy lần, với filter nào. Chênh
> lệch chất lượng nằm ở chỗ đó, và notebook này đo nó.

Planner ở đây là **rule-based, không gọi LLM** — lab chạy zero-key. Bài học là
về *chiến lược truy xuất* (tách câu hỏi, suy ra filter, phản tỉnh, thử lại),
và bài học đó rõ hơn khi planner có thể đọc được từng dòng.

In [1]:
import _setup  # noqa: F401
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

from app.agent import (SEARCH_TOOL, Agent, RetrievalTool, RuleBasedPlanner,
                       SingleShotPlanner, build_context)
from app.filters import FilteredIndex
from app.search import Searcher

DATA = Path(_setup.__file__).resolve().parent.parent / "data"

## 1. Tool definition — thứ agent thực sự nhìn thấy

Đây là toàn bộ thông tin agent có khi quyết định *có gọi retrieval hay không*.
Nếu `description` mơ hồ, agent gọi sai lúc — và không có log nào nói cho bạn biết.

In [2]:
print(json.dumps(SEARCH_TOOL, ensure_ascii=False, indent=2)[:900])

{
  "name": "search_docs",
  "description": "Search the internal Vietnamese technical documentation corpus. Covers cloud, ai_ml, security, database, networking, devops, mobile, frontend, backend and data_eng. Ask ONE question per call. Returns ranked document chunks with doc_id, title and score.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "One single-intent question."
      },
      "topic": {
        "type": "string",
        "description": "Restrict to one topic cluster.",
        "enum": [
          "cloud",
          "ai_ml",
          "security",
          "database",
          "networking",
          "devops",
          "mobile",
          "frontend",
          "backend",
          "data_eng"
        ]
      },
      "since_year": {
        "type": "integer",
        "description": "Only docs publ


Hai chi tiết đáng chú ý:

* `description` **chính là prompt truy xuất**. Nó không phải comment cho người đọc.
* `topic` là **`enum`**, không phải string tự do. Agent không thể bịa ra một topic
  không tồn tại → filter không bao giờ âm thầm khớp 0 document.

In [3]:
searcher = Searcher.from_corpus(DATA / "corpus_vn.jsonl")
index = FilteredIndex.from_searcher(searcher)
tool = RetrievalTool(index)

## 2. Planner: một câu hỏi, mấy ý định?

Câu hỏi thật của người dùng thường có **nhiều hơn một ý định**. Một embedding
duy nhất của câu ghép sẽ rơi vào *khoảng giữa* hai cụm — gần cả hai, thuộc về
không cụm nào.

In [4]:
planner = RuleBasedPlanner(budget=16)
demo_q = "tự động mở rộng theo lưu lượng và cân bằng tải giữa nhiều region"
for i, args in enumerate(planner.plan(demo_q), 1):
    print(f"  call {i}: {args.as_dict()}")

  call 1: {'query': 'tự động mở rộng theo lưu lượng', 'top_k': 8, 'topic': 'cloud'}
  call 2: {'query': 'cân bằng tải giữa nhiều region', 'top_k': 8, 'topic': 'networking'}


## 3. Đo: single-shot vs agentic, **cùng ngân sách truy xuất**

Đây là chỗ dễ đo gian lận nhất. Nếu agent được lấy 32 doc còn single-shot chỉ
16, agent thắng vì *ngân sách*, không phải vì *chiến lược*. Ở đây cả hai đều
lấy đúng **16 document** — chỉ khác cách chia.

Ngoài `recall`, ta đo thêm **`balance`**: trong 16 doc lấy về, hai vế của câu
hỏi được phủ đều đến đâu (1.00 = đều hoàn hảo, 0.00 = bỏ hẳn một vế).

In [5]:
queries = [json.loads(l) for l in (DATA / "agent_queries.jsonl").open(encoding="utf-8")]
BUDGET = 16


def evaluate(agent, label):
    rec, bal, calls, ms = [], [], [], []
    for q in queries:
        r = agent.answer(q["question"])
        truth, got = set(q["relevant_doc_ids"]), set(r.doc_ids)
        rec.append(len(truth & got) / len(truth))
        a, b = len(set(q["gold_a"]) & got), len(set(q["gold_b"]) & got)
        bal.append(min(a, b) / max(1, max(a, b)))
        calls.append(r.n_calls)
        ms.append(r.latency_ms)
    n = len(queries)
    print(f"{label:<14}{sum(rec)/n:8.3f}{sum(bal)/n:9.2f}{sum(calls)/n:8.1f}{sum(ms)/n:9.1f}")
    return sum(rec) / n


print(f"{'strategy':<20}{'recall':>8}{'balance':>9}{'calls':>8}{'ms':>9}")
base = evaluate(Agent(tool, SingleShotPlanner(budget=BUDGET)), "single-shot")
split = evaluate(Agent(tool, RuleBasedPlanner(budget=BUDGET, use_filters=False)),
                 "agentic (no filter)")
filt = evaluate(Agent(tool, RuleBasedPlanner(budget=BUDGET, use_filters=True)),
                "agentic (+filter)")
print(f"\nΔ recall vs single-shot:  tách câu {split - base:+.3f}   tách + filter {filt - base:+.3f}")

strategy              recall  balance   calls       ms
single-shot      0.526     0.08     1.0      8.6


agentic (no filter)   0.906     0.93     2.3     15.9


agentic (+filter)   0.823     0.76     2.3     30.5

Δ recall vs single-shot:  tách câu +0.380   tách + filter +0.297


**Đọc kết quả.** `balance` của single-shot rất thấp: nó gần như chỉ lấy *một*
vế của câu hỏi. Đó chính xác là lý do RAG một-lượt trả lời "đúng một nửa" cho
câu hỏi ghép — và vì câu trả lời nghe vẫn trôi chảy, lỗi này rất khó phát hiện.

Nhưng hãy nhìn cột `calls` và `ms`: agentic tốn **nhiều lần gọi hơn** và chậm
hơn tương ứng. Ở đây mỗi call chỉ là vector search; trong hệ thật mỗi vòng
planning là một **LLM call** — tức là tiền và latency thật.

> Quy tắc rút ra: **classic RAG cho tra cứu đơn giản, agentic cho câu hỏi
> nhiều phần.** Đừng bật agentic cho mọi query.

**Và hãy so hai dòng agentic với nhau.** Bật filter suy đoán làm *giảm* recall
so với chỉ tách câu — vì topic đoán từ keyword loại bỏ luôn những document liên
quan nằm ở cụm bên cạnh. Đổi lại, nó tốn ít call hơn. Đây đúng là bài học của
NB5 lặp lại ở tầng agent: **filter không miễn phí, phải đo chứ đừng đoán.**

### Soi kỹ: vì sao `agentic (+filter)` thấp hơn `agentic (no filter)`?

Doc vàng của mỗi vế nằm trọn trong **một** cụm topic (`topics` trong
`agent_queries.jsonl`). Nếu planner đoán đúng topic thì filter không thể làm
mất doc vàng — vậy recall chỉ giảm được khi planner **đoán sai topic**.
Kiểm chứng trên từng sub-question:

In [6]:
from app.agent import TOPIC_HINTS  # noqa: E402

nofilt = Agent(tool, RuleBasedPlanner(budget=BUDGET, use_filters=False))
filt = Agent(tool, RuleBasedPlanner(budget=BUDGET, use_filters=True))


def topic_of_part(part: str, q: dict) -> str:
    """Topic thật của một mảnh câu = topic của sub_question chứa mảnh đó."""
    for sub, t in zip(q["sub_questions"], q["topics"]):
        if part in sub or sub in part:
            return t
    return "?"


print(f"{'id':<8}{'recall no-filter':>17}{'recall +filter':>16}    filter đoán sai")
for q in queries:
    truth = set(q["relevant_doc_ids"])
    r0 = len(truth & set(nofilt.answer(q["question"]).doc_ids)) / len(truth)
    r1 = len(truth & set(filt.answer(q["question"]).doc_ids)) / len(truth)
    wrong = []
    for args in filt.planner.plan(q["question"]):
        true_t = topic_of_part(args.query, q)
        if args.topic and args.topic != true_t:
            kw = [h for h in TOPIC_HINTS[args.topic] if h in args.query.lower()]
            got = tool(args).doc_ids
            in_true = sum(d.startswith(true_t + "_") for d in got)
            wrong.append(f"'{args.query}' → {args.topic} (thật: {true_t}; khớp {kw}; "
                         f"{in_true}/{len(got)} doc đúng cụm)")
    flag = " ↓" if r1 < r0 else "  "
    print(f"{q['query_id']:<8}{r0:>17.3f}{r1:>16.3f}{flag}  {'; '.join(wrong)}")

id       recall no-filter  recall +filter    filter đoán sai
mq_000              1.000           1.000    
mq_001              1.000           1.000    
mq_002              0.750           0.750    


mq_003              1.000           0.500 ↓  'circuit breaker tránh cascading failure' → ai_ml (thật: backend; khớp ['ai']; 0/8 doc đúng cụm)
mq_004              0.750           0.750    
mq_005              1.000           1.000    
mq_006              1.000           1.000    
mq_007              1.000           0.500 ↓  'xác thực hai yếu tố cho người dùng' → ai_ml (thật: security; khớp ['ai']; 0/8 doc đúng cụm)


mq_008              1.000           1.000    
mq_009              1.000           1.000    
mq_010              0.625           0.625    
mq_011              0.750           0.750    


**Kết luận (theo số đo ở trên).** Cùng ngân sách 16 doc và cùng 2.3 call/câu,
`agentic (+filter)` thua `no filter` 0.083 recall (0.906 → 0.823) và 0.17
balance (0.93 → 0.76). **Toàn bộ** khoảng chênh nằm ở 2/12 câu — `mq_003` và
`mq_007`, mỗi câu mất đúng 0.5 recall (12 × 0.083 ≈ 2 × 0.5):

1. `detect_topic` so keyword bằng **substring**, nên hint `"ai"` khớp nhầm vào
   "f**ai**lure" và "h**ai** yếu tố" → planner gắn `topic=ai_ml` cho hai vế
   thực ra thuộc `backend` và `security`.
2. Filter là ràng buộc **cứng**: 0/8 doc trả về thuộc cụm đúng → vế đó mất
   trắng. Ở 10 câu còn lại recall y hệt — filter đoán đúng cũng không *thêm*
   được gì, vì search không filter vốn đã tìm đúng cụm.
3. Phản tỉnh **không cứu được**: nó chỉ chạy khi một call trả < 4 doc, còn
   filter sai cụm vẫn trả đủ 8 doc trông hợp lệ → lỗi im lặng.

Filter còn tốn thêm: cùng số call nhưng chậm gần gấp đôi (cột `ms`), vì Qdrant
chế độ in-memory kiểm tra filter trên từng điểm. Cách sửa: khớp keyword theo
ranh giới từ (`\bai\b`), chỉ filter khi chắc chắn, và phản tỉnh theo *điểm
similarity* chứ không chỉ theo *số lượng* doc.

## 4. Reflection: filter tồi còn tệ hơn không filter

`Agent` thử lại **một lần** với filter được nới ra khi một call trả về quá ít
bằng chứng. Không có bước này, một filter đoán sai sẽ âm thầm trả về rỗng.

In [7]:
from app.agent import ToolArgs  # noqa: E402

Q = "cân bằng tải giữa nhiều region"

starving = ToolArgs(query=Q, topic="networking", since_year=2027, top_k=8)
print("filter quá chặt (since_year=2027) →", len(tool(starving).doc_ids), "kết quả")

sane = ToolArgs(query=Q, topic="networking", top_k=8)
print("filter hợp lý                     →", len(tool(sane).doc_ids), "kết quả")


class StarvingPlanner:
    """Cố tình sinh ra một filter không khớp gì, để xem agent xử lý thế nào."""

    def plan(self, question):
        return [ToolArgs(query=question, topic="networking", since_year=2027, top_k=8)]


res = Agent(tool, StarvingPlanner(), min_evidence=4).answer(Q)
print(f"\nagent phản tỉnh: {res.n_calls} call → {len(res.doc_ids)} doc")
for c in res.trace:
    print("   ", c.args, "→", len(c.doc_ids), "kết quả")

filter quá chặt (since_year=2027) → 0 kết quả
filter hợp lý                     → 8 kết quả

agent phản tỉnh: 2 call → 8 doc
    {'query': 'cân bằng tải giữa nhiều region', 'top_k': 8, 'topic': 'networking', 'since_year': 2027} → 0 kết quả
    {'query': 'cân bằng tải giữa nhiều region', 'top_k': 8} → 8 kết quả


## 5. Ghép ngữ cảnh: nơi feature store gặp vector store

`build_context()` hỏi **hai câu hỏi khác nhau**:

* **Feature store** → *"user này là ai"* (cá nhân hoá, online lookup <10 ms)
* **Vector store** → *"cái gì liên quan"* (grounding)

Chạy được cả khi chưa `feast apply` (NB4) — lúc đó `features` rỗng và agent
chỉ còn grounding. Chạy NB4 trước rồi quay lại đây để thấy khác biệt.

In [8]:
store = None
try:
    from feast import FeatureStore
    repo = Path(_setup.__file__).resolve().parent.parent / "app" / "feast_repo"
    if (repo / "registry.db").exists():
        store = FeatureStore(repo_path=str(repo))
except Exception as exc:  # noqa: BLE001
    print("Feast chưa sẵn sàng:", exc)

ctx = build_context("u_001", "làm sao tối ưu chi phí hạ tầng", tool, feature_store=store)
print("features   :", ctx["features"] or "(chưa có — chạy NB4 trước)")
print("affinity   :", ctx["affinity_used"])
print("tool_args  :", ctx["tool_args"])
print("doc_ids    :", ctx["doc_ids"][:5], "…")

features   : {'user_id': ['u_001'], 'topic_affinity': ['cloud'], 'preferred_language': ['vi']}
affinity   : cloud
tool_args  : {'query': 'làm sao tối ưu chi phí hạ tầng', 'top_k': 8, 'topic': 'cloud'}
doc_ids    : ['cloud_066', 'cloud_065', 'cloud_062', 'cloud_004', 'cloud_088'] …


## Deliverable evidence

1. §1: tool schema in ra, thấy rõ `description` + `enum` của `topic`.
2. §3: bảng single-shot vs agentic — Δrecall và Δbalance dương, kèm chi phí calls/ms.
3. §4: agent phục hồi được sau một filter sai.
4. §5: `build_context()` trả về cả feature lẫn doc_ids.

---

## Vibe-coding callout

**Delegate freely:** JSON schema của tool, vòng lặp đánh giá, code in bảng.
AI viết nhanh và đúng.

**Think hard yourself:** *ngân sách so sánh*. Khi bạn bảo AI "so sánh agentic
với single-shot", nó gần như luôn cho agent gọi nhiều lần với `top_k` giữ
nguyên — nghĩa là agent lấy về gấp đôi số document. Kết quả: agent "thắng"
một cách vô nghĩa. Bài đo chỉ có giá trị khi **tổng số document lấy về bằng
nhau**. Hãy tự kiểm tra dòng `per = budget // len(parts)` trước khi tin bất kỳ
con số nào ở §3.